# 1. Análise de desvios produtivos

## 1.1 Objetivo

Esta etapa tem como objetivo investigar a utilização das previsões
produzidas pelo modelo Gradient Boosting C2 como suporte ao
acompanhamento da produção leiteira.

Serão utilizadas as 189 previsões do conjunto independente de teste,
composto por animais que não participaram do treinamento.

A análise compara a produção efetivamente registrada com a produção
estimada pelo modelo, identificando diferenças positivas e negativas.

Inicialmente, serão examinados os maiores desvios negativos, com o
objetivo de verificar como essas informações podem contribuir para
priorizar animais que mereçam acompanhamento.

Os desvios representam diferenças entre valores observados e
estimados, não constituindo diagnóstico de doenças ou identificação
automática das causas de alterações produtivas.

In [ ]:
import pandas as pd
from pathlib import Path

# Carregar as previsões do teste final
caminho = Path("../resultados/predicoes_teste_modelo_final.csv")

previsoes = pd.read_csv(
    caminho,
    parse_dates=["data_prova"]
)

# Calcular os desvios
previsoes["desvio_kg"] = (
    previsoes["kg_leite"] - previsoes["kg_previsto"]
)

# Calcular diferença percentual em relação à previsão
previsoes["desvio_percentual"] = (
    previsoes["desvio_kg"]
    / previsoes["kg_previsto"].where(previsoes["kg_previsto"] > 0)
) * 100

# Identificar apenas a direção do desvio
previsoes["situacao"] = previsoes["desvio_kg"].apply(
    lambda x: "Abaixo da estimativa" if x < 0
    else "Acima ou igual à estimativa"
)

print("Total de controles analisados:", len(previsoes))

print(
    "Controles abaixo da estimativa:",
    (previsoes["desvio_kg"] < 0).sum()
)

print(
    "Controles acima ou iguais à estimativa:",
    (previsoes["desvio_kg"] >= 0).sum()
)

print("\n10 maiores desvios negativos:\n")

colunas = [
    "propriedade",
    "id_animal_global",
    "data_prova",
    "kg_leite",
    "kg_previsto",
    "desvio_kg",
    "desvio_percentual",
    "producao_anterior",
    "del_harmonizado"
]

display(
    previsoes.loc[previsoes["desvio_kg"] < 0]
    .nsmallest(10, "desvio_kg")[colunas]
    .round(2)
)

## 1.2 Análise individual de desvios produtivos

Nesta etapa, será analisado o histórico de produção
de um animal que apresentou desvio negativo expressivo
entre a produção observada e a estimada.

O objetivo é verificar o comportamento dos controles
anteriores e identificar se a diferença observada
corresponde a uma alteração no histórico produtivo
ou a uma possível limitação da previsão.

A análise é exploratória e não permite determinar
a causa biológica de uma eventual queda de produção.

In [ ]:
import matplotlib.pyplot as plt

# Animal selecionado para investigação
animal = "CC_FESTA 1106"

# Recuperar os controles desse animal
historico = (
    previsoes.loc[
        previsoes["id_animal_global"] == animal,
        [
            "data_prova",
            "kg_leite",
            "kg_previsto",
            "producao_anterior",
            "del_harmonizado",
            "desvio_kg",
            "desvio_percentual"
        ]
    ]
    .sort_values("data_prova")
    .copy()
)

print("Animal:", animal)
print("Quantidade de controles:", len(historico))

display(
    historico.round({
        "kg_leite": 2,
        "kg_previsto": 2,
        "producao_anterior": 2,
        "del_harmonizado": 2,
        "desvio_kg": 2,
        "desvio_percentual": 2
    })
)

# Visualizar produção real e prevista
plt.figure(figsize=(10, 5))

plt.plot(
    historico["data_prova"],
    historico["kg_leite"],
    marker="o",
    label="Produção real"
)

plt.plot(
    historico["data_prova"],
    historico["kg_previsto"],
    marker="s",
    linestyle="--",
    label="Produção prevista"
)

plt.title(f"Histórico produtivo — {animal}")
plt.xlabel("Data do controle")
plt.ylabel("Produção de leite (kg)")

plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Comparar médias reais e previstas por propriedade

resumo = (
    previsoes.groupby("propriedade")
    .agg(
        controles=("kg_leite", "size"),
        producao_real_media=("kg_leite", "mean"),
        producao_prevista_media=("kg_previsto", "mean"),
        erro_medio=("erro", "mean"),
        mae=("erro_absoluto", "mean")
    )
    .round(2)
)

print("Comparação por propriedade:")
display(resumo)

# Verificar as 10 maiores previsões do modelo

print("\n10 maiores previsões realizadas:")

colunas = [
    "propriedade",
    "id_animal_global",
    "data_prova",
    "producao_anterior",
    "del_harmonizado",
    "kg_leite",
    "kg_previsto",
    "erro"
]

display(
    previsoes.nlargest(10, "kg_previsto")[colunas]
    .round({
        "producao_anterior": 2,
        "del_harmonizado": 2,
        "kg_leite": 2,
        "kg_previsto": 2,
        "erro": 2
    })
)

In [ ]:
import numpy as np

# Criar faixas com base na produção prevista
diagnostico = previsoes.copy()

diagnostico["faixa_prevista"] = pd.cut(
    diagnostico["kg_previsto"],
    bins=[-np.inf, 15, 25, 35, 45, np.inf],
    labels=[
        "Até 15 kg",
        "15–25 kg",
        "25–35 kg",
        "35–45 kg",
        "Acima de 45 kg"
    ]
)

# Identificar superestimações individuais
diagnostico["superestimou"] = (
    diagnostico["kg_previsto"] > diagnostico["kg_leite"]
)

# Avaliar o desempenho por propriedade e faixa
resumo_faixas = (
    diagnostico
    .groupby(
        ["propriedade", "faixa_prevista"],
        observed=True
    )
    .agg(
        controles=("kg_leite", "size"),
        real_medio=("kg_leite", "mean"),
        previsto_medio=("kg_previsto", "mean"),
        erro_medio=("erro", "mean"),
        mae=("erro_absoluto", "mean"),
        proporcao_superestimada=("superestimou", "mean")
    )
    .reset_index()
)

resumo_faixas["proporcao_superestimada"] *= 100

display(
    resumo_faixas.round({
        "real_medio": 2,
        "previsto_medio": 2,
        "erro_medio": 2,
        "mae": 2,
        "proporcao_superestimada": 1
    })
)

In [ ]:
import numpy as np
import pandas as pd

# Selecionar controles com produção anterior disponível
comparacao = previsoes.loc[
    previsoes["producao_anterior"].notna()
    & (previsoes["producao_anterior"] > 0)
].copy()

# Estratégia simples: prever o mesmo valor do controle anterior
comparacao["previsao_simples"] = comparacao["producao_anterior"]

# Calcular erros das duas estratégias
comparacao["erro_ml_abs"] = (
    comparacao["kg_leite"] - comparacao["kg_previsto"]
).abs()

comparacao["erro_simples_abs"] = (
    comparacao["kg_leite"] - comparacao["previsao_simples"]
).abs()

# Resumo geral
mae_ml = comparacao["erro_ml_abs"].mean()
mae_simples = comparacao["erro_simples_abs"].mean()

rmse_ml = np.sqrt(
    ((comparacao["kg_leite"] - comparacao["kg_previsto"]) ** 2).mean()
)

rmse_simples = np.sqrt(
    ((comparacao["kg_leite"] - comparacao["previsao_simples"]) ** 2).mean()
)

print("Controles comparados:", len(comparacao))

print("\nGradient Boosting:")
print(f"MAE: {mae_ml:.3f} kg")
print(f"RMSE: {rmse_ml:.3f} kg")

print("\nProdução anterior como previsão:")
print(f"MAE: {mae_simples:.3f} kg")
print(f"RMSE: {rmse_simples:.3f} kg")

print("\nComparação por propriedade:")

resumo_comparacao = (
    comparacao.groupby("propriedade")
    .agg(
        controles=("kg_leite", "size"),
        mae_gradient_boosting=("erro_ml_abs", "mean"),
        mae_producao_anterior=("erro_simples_abs", "mean")
    )
    .round(3)
)

display(resumo_comparacao)

## 1.5 Resumo de desvios produtivos por animal

Esta etapa organiza as previsões do conjunto de teste
por animal, com o objetivo de demonstrar uma possível
funcionalidade de acompanhamento produtivo.

São contabilizados os controles com produção inferior
à estimativa e aqueles que apresentam simultaneamente
uma redução em relação à produção anterior.

Também é identificado o maior desvio negativo de cada
animal no conjunto analisado.

Os resultados constituem indicadores exploratórios.
A presença de desvios não comprova doença, alteração
de manejo ou qualquer outra causa específica.

In [ ]:
from pathlib import Path

# Preparar os dados de acompanhamento
acompanhamento = previsoes.copy()

acompanhamento["desvio_kg"] = (
    acompanhamento["kg_leite"]
    - acompanhamento["kg_previsto"]
)

acompanhamento["variacao_anterior_kg"] = (
    acompanhamento["kg_leite"]
    - acompanhamento["producao_anterior"]
)

# Verificar as duas condições separadamente
acompanhamento["abaixo_estimativa"] = (
    acompanhamento["desvio_kg"] < 0
)

acompanhamento["queda_anterior"] = (
    acompanhamento["producao_anterior"].notna()
    & (acompanhamento["producao_anterior"] > 0)
    & (acompanhamento["variacao_anterior_kg"] < 0)
)

# Controles que apresentam as duas condições
acompanhamento["duas_condicoes"] = (
    acompanhamento["abaixo_estimativa"]
    & acompanhamento["queda_anterior"]
)

# Resumo individual
resumo_animais = (
    acompanhamento
    .groupby(
        ["propriedade", "id_animal_global"],
        as_index=False
    )
    .agg(
        controles_teste=("kg_leite", "size"),
        controles_abaixo_estimativa=("abaixo_estimativa", "sum"),
        controles_com_duas_condicoes=("duas_condicoes", "sum"),
        maior_desvio_negativo_kg=("desvio_kg", "min")
    )
)

# Manter animais com pelo menos um desvio negativo
resumo_animais = resumo_animais.loc[
    resumo_animais["controles_abaixo_estimativa"] > 0
].copy()

# Ordenar pelo maior desvio negativo observado
resumo_animais = resumo_animais.sort_values(
    "maior_desvio_negativo_kg"
)

print("Animais com pelo menos um controle abaixo da estimativa:")
print(len(resumo_animais))

print("\n10 animais com maiores desvios negativos observados:")

display(
    resumo_animais.head(10).round(2)
)

# Salvar os resultados exploratórios
saida = Path("../resultados/resumo_animais_apoio_decisao.csv")

resumo_animais.to_csv(
    saida,
    index=False
)

print("\nArquivo salvo em:", saida)

## 1.6 Visualização dos maiores desvios produtivos

Esta etapa apresenta graficamente os maiores desvios negativos
identificados nas previsões do conjunto independente de teste.

Para cada animal, considera-se o controle que apresentou a
maior diferença negativa entre produção real e estimada.

A visualização permite organizar os registros que merecem
revisão, demonstrando uma possível aplicação do modelo
no apoio ao acompanhamento produtivo.

Os resultados não representam diagnóstico clínico,
classificação de risco ou identificação das causas
das reduções na produção.

In [ ]:
import matplotlib.pyplot as plt

# Selecionar os 10 animais com maiores desvios negativos
ranking = (
    resumo_animais
    .nsmallest(10, "maior_desvio_negativo_kg")
    .copy()
)

# Representar a magnitude do desvio como valor positivo
ranking["magnitude_desvio_kg"] = (
    -ranking["maior_desvio_negativo_kg"]
)

# Simplificar nomes para visualização
ranking["animal"] = (
    ranking["id_animal_global"]
    .str.replace(r"^CC_", "", regex=True)
)

# Ordenar do maior para o menor desvio
ranking = ranking.sort_values(
    "magnitude_desvio_kg",
    ascending=False
)

fig, ax = plt.subplots(figsize=(11, 6))

barras = ax.barh(
    ranking["animal"],
    ranking["magnitude_desvio_kg"]
)

ax.invert_yaxis()

ax.bar_label(
    barras,
    labels=[
        f"{valor:.2f} kg"
        for valor in ranking["magnitude_desvio_kg"]
    ],
    padding=5
)

# Criar espaço para os valores ao final das barras
ax.set_xlim(
    0,
    ranking["magnitude_desvio_kg"].max() * 1.25
)

ax.set_title(
    "Maiores diferenças entre produção prevista e real",
    fontsize=14,
    fontweight="bold",
    pad=18
)

ax.set_xlabel(
    "Magnitude do desvio negativo (kg)"
)

ax.set_ylabel("Animal")

ax.grid(
    axis="x",
    alpha=0.2
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 1.7 Análise temporal da produção leiteira

Esta etapa investiga a evolução da produção real dos animais
ao longo dos controles disponíveis no conjunto de teste.

Para cada animal, os registros serão organizados
cronologicamente, permitindo comparar a produção atual
com a produção do controle anterior.

Serão calculadas a variação absoluta, a variação
percentual e a quantidade de dias entre os controles.

Essas informações serão analisadas em conjunto com
os desvios das previsões do Gradient Boosting.

A análise é retrospectiva e exploratória. Intervalos
longos entre controles ou mudanças de lactação podem
comprometer a comparação direta entre registros.

In [ ]:
# Organizar os controles cronologicamente por animal
historico_temporal = (
    previsoes
    .sort_values(["id_animal_global", "data_prova"])
    .copy()
)

# Agrupar os registros por animal
grupo = historico_temporal.groupby("id_animal_global")

# Recuperar os controles anteriores presentes no teste
historico_temporal["data_anterior"] = (
    grupo["data_prova"].shift(1)
)

historico_temporal["kg_anterior_no_teste"] = (
    grupo["kg_leite"].shift(1)
)

historico_temporal["del_anterior_no_teste"] = (
    grupo["del_harmonizado"].shift(1)
)

# Calcular intervalo entre controles
historico_temporal["dias_entre_controles"] = (
    historico_temporal["data_prova"]
    - historico_temporal["data_anterior"]
).dt.days

# Calcular a variação real entre controles
historico_temporal["variacao_real_kg"] = (
    historico_temporal["kg_leite"]
    - historico_temporal["kg_anterior_no_teste"]
)

historico_temporal["variacao_real_percentual"] = (
    historico_temporal["variacao_real_kg"]
    / historico_temporal["kg_anterior_no_teste"].where(
        historico_temporal["kg_anterior_no_teste"] > 0
    )
) * 100

# Verificar quando o DEL indica possível reinício de lactação
historico_temporal["possivel_reinicio_lactacao"] = (
    historico_temporal["del_harmonizado"].notna()
    & historico_temporal["del_anterior_no_teste"].notna()
    & (
        historico_temporal["del_harmonizado"]
        < historico_temporal["del_anterior_no_teste"]
    )
)

# Identificar queda real e desvio negativo no mesmo controle
historico_temporal["queda_e_desvio_negativo"] = (
    (historico_temporal["variacao_real_kg"] < 0)
    & (historico_temporal["kg_leite"] < historico_temporal["kg_previsto"])
    & (~historico_temporal["possivel_reinicio_lactacao"])
)

# Selecionar três animais para investigação
animais_analisados = [
    "CC_FESTA 1106",
    "CC_JADE-1270",
    "CC_GIOCONDA-10765"
]

colunas = [
    "id_animal_global",
    "data_prova",
    "kg_anterior_no_teste",
    "kg_leite",
    "kg_previsto",
    "variacao_real_kg",
    "variacao_real_percentual",
    "desvio_kg",
    "dias_entre_controles",
    "possivel_reinicio_lactacao",
    "queda_e_desvio_negativo"
]

resultado = historico_temporal.loc[
    historico_temporal["id_animal_global"].isin(
        animais_analisados
    ),
    colunas
]

display(
    resultado.round({
        "kg_anterior_no_teste": 2,
        "kg_leite": 2,
        "kg_previsto": 2,
        "variacao_real_kg": 2,
        "variacao_real_percentual": 2,
        "desvio_kg": 2
    })
)

In [ ]:
animais = [
    "CC_FESTA 1106",
    "CC_GIOCONDA-10765",
    "CC_JADE-1270"
]

conferencia = (
    historico_temporal.loc[
        historico_temporal["id_animal_global"].isin(animais),
        [
            "id_animal_global",
            "data_prova",
            "del_harmonizado",
            "kg_leite",
            "producao_anterior"
        ]
    ]
    .sort_values(["id_animal_global", "data_prova"])
    .copy()
)

grupo = conferencia.groupby("id_animal_global")

conferencia["dias_entre_controles"] = (
    grupo["data_prova"].diff().dt.days
)

conferencia["variacao_del"] = (
    grupo["del_harmonizado"].diff()
)

print(conferencia.to_string(index=False))

## 1.8 Demonstração de acompanhamento produtivo

Esta etapa apresenta uma demonstração exploratória de
um sistema de apoio à decisão baseado nas previsões
do Gradient Boosting e no histórico dos animais.

São apresentados três exemplos de comportamento
produtivo: redução abrupta, reduções consecutivas
e recuperação após uma queda.

A visualização combina produção real, produção
estimada e dias em lactação.

O objetivo é demonstrar como o acompanhamento
histórico pode complementar as previsões de
Aprendizado de Máquina.

Os exemplos não constituem diagnóstico de problemas
sanitários ou nutricionais.

In [ ]:
import matplotlib.pyplot as plt

animais_demo = [
    ("CC_FESTA 1106", "FESTA 1106 — Queda abrupta"),
    ("CC_GIOCONDA-10765", "GIOCONDA 10765 — Reduções consecutivas"),
    ("CC_JADE-1270", "JADE 1270 — Recuperação produtiva")
]

fig, axes = plt.subplots(
    3, 1,
    figsize=(12, 13)
)

for ax, (id_animal, titulo) in zip(axes, animais_demo):

    dados = (
        historico_temporal.loc[
            historico_temporal["id_animal_global"] == id_animal
        ]
        .sort_values("data_prova")
        .copy()
    )

    ax.plot(
        dados["data_prova"],
        dados["kg_leite"],
        marker="o",
        linewidth=2,
        label="Produção real"
    )

    ax.plot(
        dados["data_prova"],
        dados["kg_previsto"],
        marker="s",
        linestyle="--",
        linewidth=2,
        label="Produção prevista"
    )

    # Destacar os controles reais com valores
    for _, linha in dados.iterrows():
        ax.annotate(
            f'{linha["kg_leite"]:.1f} kg',
            (
                linha["data_prova"],
                linha["kg_leite"]
            ),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=9
        )

    ax.set_title(
        titulo,
        fontweight="bold",
        fontsize=12
    )

    ax.set_ylabel("Produção (kg)")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")

    # Mostrar somente datas existentes nos registros
    ax.set_xticks(dados["data_prova"])
    ax.set_xticklabels(
        dados["data_prova"].dt.strftime("%d/%m/%Y"),
        rotation=30,
        ha="right"
    )

axes[-1].set_xlabel("Data do controle")

fig.suptitle(
    "Demonstração de acompanhamento produtivo",
    fontsize=16,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()

# Salvar figura para apresentação ao orientador
from pathlib import Path

saida = Path("../resultados")

saida.mkdir(parents=True, exist_ok=True)

fig.savefig(
    saida / "demonstracao_apoio_decisao.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figura salva em:")
print(saida / "demonstracao_apoio_decisao.png")